# 03 — Model Training Orchestrator  —  VERSION GOOGLE COLAB (single GPU)
## Hackathon IBM — Détection de Fraude Bancaire

**Notebook prêt-à-run sur Colab T4/L4/A100**. Entraîne CatBoost / XGBoost / TabNet sur tes datasets préparés et logue tout.

### Workflow
1. **Cellule 0-1** : Installation + vérification GPU (`nvidia-smi` + tests CUDA).
2. **Cellule 2** : Upload / localisation des fichiers `prepared_*.parquet`.
3. **Cellules 3-10** : Définitions (modulaire, identique au notebook workbench).
4. **Cellules 11-13** : Boucle d'expérimentation avec **barre de progression en %** et **monitoring VRAM live**.
5. **Cellules 14-16** : Leaderboard + graphiques.

### Ce qui est spécifique à Colab
- Cellules `!nvidia-smi` et `!pip install` intégrées.
- **Test benchmark CPU vs GPU** qui confirme que XGBoost utilise bien le GPU.
- **Monitoring VRAM** toutes les 3 s pendant les entraînements (thread background).
- **Progress bar double** : globale (fichiers×modèles) + interne (itérations du modèle).

---
## 0. Activer le GPU sur Colab

**IMPORTANT** : avant tout, dans Colab → `Runtime` → `Change runtime type` → `Hardware accelerator: GPU` (T4 ou L4 suffit largement).

In [ ]:
!nvidia-smi

Tu dois voir quelque chose comme `Tesla T4`, `L4` ou `A100` avec une ligne `0 MiB / 15360 MiB` de VRAM utilisée. **Si la commande ne retourne rien**, le GPU n'est PAS activé — retourne dans le menu Runtime et active-le.

---
## 1. Installation des libs

In [ ]:
# Colab a déjà torch+CUDA, pandas, numpy, sklearn, matplotlib.
# On n'installe que ce qui manque.
!pip install -q catboost xgboost pytorch-tabnet pyarrow tqdm

---
## 2. Imports & vérification CUDA

In [ ]:
import os, re, gc, time, glob, json, threading, warnings, subprocess
from pathlib import Path
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd

from sklearn.metrics import (f1_score, recall_score, precision_score, accuracy_score,
                              roc_auc_score, average_precision_score, confusion_matrix)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import catboost as cb
import xgboost as xgb
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

CUDA_AVAILABLE = torch.cuda.is_available()
N_GPUS = torch.cuda.device_count() if CUDA_AVAILABLE else 0
DEVICE = 'cuda' if CUDA_AVAILABLE else 'cpu'
XGB_VERSION = tuple(int(x) for x in xgb.__version__.split('.')[:2])

print('=== Environment ===')
print(f'pandas   : {pd.__version__}')
print(f'torch    : {torch.__version__}')
print(f'catboost : {cb.__version__}')
print(f'xgboost  : {xgb.__version__}')
print(f'CUDA     : {CUDA_AVAILABLE} (devices={N_GPUS})')
if CUDA_AVAILABLE:
    for i in range(N_GPUS):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU[{i}] : {props.name}  |  VRAM total : {props.total_memory/1024**3:.1f} GB')
else:
    raise RuntimeError('❌ CUDA NON DISPONIBLE. Active le GPU dans Runtime → Change runtime type.')

---
## 3. TEST de vérification : est-ce que le GPU est vraiment utilisé par XGBoost ?

On fait un **mini benchmark CPU vs GPU** sur un dataset synthétique. Si XGBoost GPU fonctionne, le run GPU doit être **nettement plus rapide** et on doit voir la VRAM augmenter sur `nvidia-smi`.

In [ ]:
from sklearn.datasets import make_classification

print('Génération d\'un dataset synthétique (100k × 30)...')
X_bench, y_bench = make_classification(n_samples=100_000, n_features=30, n_informative=20,
                                         n_redundant=5, weights=[0.98, 0.02], random_state=42)
X_bench = pd.DataFrame(X_bench.astype(np.float32))

# --- CPU run ---
m_cpu = xgb.XGBClassifier(n_estimators=300, tree_method='hist', device='cpu',
                           max_depth=6, verbosity=0)
t0 = time.perf_counter(); m_cpu.fit(X_bench, y_bench); t_cpu = time.perf_counter() - t0
print(f'  XGBoost CPU : {t_cpu:6.2f}s')

# --- GPU run (on vérifie la VRAM avant/après pour prouver qu'elle est utilisée) ---
torch.cuda.empty_cache()
vram_before = torch.cuda.memory_allocated() / 1024**2

if XGB_VERSION >= (2, 0):
    m_gpu = xgb.XGBClassifier(n_estimators=300, tree_method='hist', device='cuda:0',
                               max_depth=6, verbosity=0)
else:
    m_gpu = xgb.XGBClassifier(n_estimators=300, tree_method='gpu_hist', gpu_id=0,
                               max_depth=6, verbosity=0)

t0 = time.perf_counter(); m_gpu.fit(X_bench, y_bench); t_gpu = time.perf_counter() - t0

# VRAM peak pendant le fit
vram_peak = torch.cuda.max_memory_allocated() / 1024**2

print(f'  XGBoost GPU : {t_gpu:6.2f}s  (VRAM peak ≈ {vram_peak:.0f} MB)')
print(f'\n  Speedup GPU / CPU : ×{t_cpu/t_gpu:.1f}')

if t_gpu < t_cpu * 0.8:
    print('\n✅ GPU BIEN UTILISÉ par XGBoost — speedup confirmé.')
else:
    print('\n⚠ Speedup faible — sur un petit dataset, l\'overhead GPU peut dominer. '
          'La vraie confirmation viendra avec le monitoring VRAM pendant les vrais runs (plus bas).')

del X_bench, y_bench, m_cpu, m_gpu
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Test CatBoost GPU (il RAISE si GPU non dispo, donc c'est une preuve directe)
print('Test CatBoost GPU...')
m = cb.CatBoostClassifier(iterations=50, task_type='GPU', devices='0', verbose=False,
                           allow_writing_files=False)
X = np.random.randn(5000, 20).astype(np.float32)
y = (np.random.rand(5000) > 0.95).astype(int)
m.fit(X, y)
print('  ✅ CatBoost GPU OK')
del m; gc.collect(); torch.cuda.empty_cache()

# Test TabNet CUDA
print('\nTest TabNet CUDA...')
from pytorch_tabnet.tab_model import TabNetClassifier as _TN
t = _TN(device_name='cuda', verbose=0, seed=42)
t.fit(X_train=X, y_train=y, max_epochs=3, patience=3, batch_size=256,
      virtual_batch_size=64)
print(f'  ✅ TabNet CUDA OK (device={t.device})')
del t, X, y; gc.collect(); torch.cuda.empty_cache()

print('\n=== Les 3 modèles sont prêts à utiliser le GPU ===')

---
## 4. Localiser les fichiers `prepared_*.parquet`

Trois options pour charger tes parquets sur Colab :

**Option A — Upload direct** (cellule ci-dessous, sélectionne les 11 fichiers depuis ton PC)

**Option B — Monter Google Drive** et ajuster le chemin :
```python
from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/hackaton_ibm')
```

**Option C — Fichiers déjà présents dans `/content`** (uploadés en avance ou clonés depuis GitHub). Dans ce cas, saute la cellule d'upload.

In [ ]:
# --- OPTION A : Upload direct (décommente si besoin) ---
# from google.colab import files
# uploaded = files.upload()   # sélectionne prepared_train_*.parquet + prepared_test_050.0_pct.parquet

# --- OPTION B : Google Drive (décommente si besoin) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/hackaton_ibm')  # ajuste ce chemin

# --- Par défaut : dossier courant /content ---
DATA_DIR = Path('.')

# Vérification
found = sorted(DATA_DIR.glob('prepared_*.parquet'))
print(f'Fichiers trouvés dans {DATA_DIR.resolve()} :')
for f in found:
    size_mb = f.stat().st_size / 1024**2
    print(f'  - {f.name:<42} ({size_mb:6.2f} MB)')

assert len(found) >= 2, (
    f'❌ Aucun parquet trouvé dans {DATA_DIR.resolve()}. '
    'Upload tes fichiers (Option A) ou monte ton Drive (Option B).'
)

---
## 5. Configuration globale

In [ ]:
TRAIN_GLOB  = 'prepared_train_*.parquet'
TEST_FILE   = 'prepared_test_050.0_pct.parquet'

LOG_DIR     = Path('logs_training_colab')
MODELS_DIR  = LOG_DIR / 'models'
RESULTS_CSV = LOG_DIR / 'experiment_results.csv'

LOG_DIR.mkdir(exist_ok=True, parents=True)
MODELS_DIR.mkdir(exist_ok=True, parents=True)

TARGET = 'is_fraud'
RANDOM_STATE = 42

# --- Budget Colab single GPU ---
CATBOOST_ITER    = 1500
XGBOOST_ITER     = 1500
TABNET_EPOCHS    = 60
TABNET_BATCH_MAX = 1024

# --- Mode run ---
# 'smoke' : 3 ratios pour tester vite
# 'full'  : les 10 ratios
RUN_MODE = 'full'
SMOKE_RATIOS = [2.0, 10.0, 30.0]

# --- Affichage pendant l'entraînement ---
# On laisse un verbose modéré pour SUIVRE que le GPU bosse réellement
CATBOOST_METRIC_PERIOD = 200   # print toutes les 200 itérations
XGBOOST_VERBOSE        = 200   # print toutes les 200 itérations

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(RANDOM_STATE)

print(f'Run mode    : {RUN_MODE}')
print(f'Logs        -> {LOG_DIR.resolve()}')
print(f'Models      -> {MODELS_DIR.resolve()}')

---
## 6. Monitor VRAM en arrière-plan

Un thread qui **log la VRAM toutes les N secondes** pendant chaque entraînement. C'est LA preuve vivante que le GPU bosse : si `allocated` reste à 0 MB, il ne bosse pas.

In [ ]:
class VRAMMonitor:
    """Thread daemon qui track la VRAM toutes les `interval` secondes.
    Usage :
        mon = VRAMMonitor(interval=3.0); mon.start()
        ... train model ...
        mon.stop()
        print(mon.report())
    """
    def __init__(self, interval: float = 3.0):
        self.interval = interval
        self._stop = threading.Event()
        self._thread = None
        self.samples = []   # list of (t_rel, allocated_mb, reserved_mb)

    def _loop(self):
        t0 = time.perf_counter()
        while not self._stop.is_set():
            try:
                alloc = torch.cuda.memory_allocated() / 1024**2
                reser = torch.cuda.memory_reserved()  / 1024**2
                self.samples.append((time.perf_counter() - t0, alloc, reser))
            except Exception:
                pass
            self._stop.wait(self.interval)

    def start(self):
        self.samples = []
        self._stop.clear()
        if CUDA_AVAILABLE:
            torch.cuda.reset_peak_memory_stats()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=self.interval + 1)

    def report(self) -> str:
        if not self.samples:
            return 'VRAM : pas d\'échantillon'
        arr = np.array(self.samples)
        peak_alloc = arr[:, 1].max()
        peak_res   = arr[:, 2].max()
        mean_alloc = arr[:, 1].mean()
        return (f'VRAM : peak_alloc={peak_alloc:.0f} MB | peak_reserved={peak_res:.0f} MB | '
                f'mean_alloc={mean_alloc:.0f} MB | n_samples={len(arr)}')

---
## 7. Helpers génériques

In [ ]:
_RATIO_RE = re.compile(r'(\d+\.\d+)')


def extract_ratio(path) -> float:
    m = _RATIO_RE.search(Path(path).stem)
    return float(m.group(1)) if m else np.nan


def load_dataset(path) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    df = pd.read_parquet(path)
    y = df[TARGET].astype(np.int8)
    X = df.drop(columns=[TARGET])
    cat_features = [c for c, dt in X.dtypes.items() if str(dt) == 'category']
    return X, y, cat_features


def align_categories(X_train, X_test, cat_features):
    X_train = X_train.copy(); X_test = X_test.copy()
    for c in cat_features:
        all_cats = pd.api.types.union_categoricals([X_train[c], X_test[c]], sort_categories=True).categories
        X_train[c] = pd.Categorical(X_train[c], categories=all_cats)
        X_test[c]  = pd.Categorical(X_test[c],  categories=all_cats)
    return X_train, X_test


def compute_metrics(y_true, y_pred, y_proba):
    metrics = {
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Accuracy':  accuracy_score(y_true, y_pred),
        'PR-AUC':    average_precision_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        'ROC-AUC':   roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
    }
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    metrics.update({'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)})
    return metrics


def append_result_row(row, csv_path):
    df_row = pd.DataFrame([row])
    df_row.to_csv(csv_path, mode='a', header=not csv_path.exists(), index=False)

---
## 8. Entraîneurs (CatBoost / XGBoost / TabNet)  — **GPU forcé + verbose**

`verbose` à une période modérée pour qu'on voie défiler les itérations (= preuve que ça bosse).

In [ ]:
def train_catboost(X_train, y_train, X_test, y_test, cat_features):
    Xtr = X_train.copy(); Xte = X_test.copy()
    for c in cat_features:
        Xtr[c] = Xtr[c].astype(str).fillna('missing')
        Xte[c] = Xte[c].astype(str).fillna('missing')

    params = dict(
        iterations=CATBOOST_ITER, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
        random_seed=RANDOM_STATE, eval_metric='PRAUC', loss_function='Logloss',
        auto_class_weights='Balanced', od_type='Iter', od_wait=100,
        verbose=CATBOOST_METRIC_PERIOD,   # print périodique (preuve d'activité)
        allow_writing_files=False,
        task_type='GPU', devices='0',     # force GPU 0 de Colab
    )

    model = cb.CatBoostClassifier(**params)
    t0 = time.perf_counter()
    model.fit(Xtr, y_train, cat_features=cat_features,
              eval_set=(Xte, y_test), use_best_model=True)
    elapsed = time.perf_counter() - t0

    y_proba = model.predict_proba(Xte)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    return model, y_pred, y_proba, elapsed


def save_catboost(model, path):
    model.save_model(str(path.with_suffix('.cbm')))

In [ ]:
def train_xgboost(X_train, y_train, X_test, y_test, cat_features):
    X_train_al, X_test_al = align_categories(X_train, X_test, cat_features)

    params = dict(
        n_estimators=XGBOOST_ITER, learning_rate=0.05, max_depth=6,
        min_child_weight=1.0, subsample=0.85, colsample_bytree=0.85,
        reg_lambda=1.0, objective='binary:logistic', eval_metric='aucpr',
        scale_pos_weight=float((y_train == 0).sum() / max((y_train == 1).sum(), 1)),
        random_state=RANDOM_STATE, enable_categorical=True,
        n_jobs=-1, early_stopping_rounds=100, verbosity=0,
    )
    # --- GPU forcé ---
    if XGB_VERSION >= (2, 0):
        params.update(dict(tree_method='hist', device='cuda:0'))
    else:
        params.update(dict(tree_method='gpu_hist', gpu_id=0, predictor='gpu_predictor'))

    model = xgb.XGBClassifier(**params)
    t0 = time.perf_counter()
    model.fit(X_train_al, y_train,
              eval_set=[(X_test_al, y_test)],
              verbose=XGBOOST_VERBOSE)    # print périodique du AUCPR
    elapsed = time.perf_counter() - t0

    y_proba = model.predict_proba(X_test_al)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    return model, y_pred, y_proba, elapsed


def save_xgboost(model, path):
    model.save_model(str(path.with_suffix('.ubj')))

In [ ]:
def build_tabnet_preprocessing(X_train, X_test, cat_features):
    num_cols = [c for c in X_train.columns if c not in cat_features]
    ordered = cat_features + num_cols
    X_tr, X_te = X_train[ordered].copy(), X_test[ordered].copy()

    cat_dims = []
    for c in cat_features:
        enc = LabelEncoder()
        combined = pd.concat([X_tr[c].astype(str), X_te[c].astype(str)])
        enc.fit(combined.fillna('missing'))
        X_tr[c] = enc.transform(X_tr[c].astype(str).fillna('missing'))
        X_te[c] = enc.transform(X_te[c].astype(str).fillna('missing'))
        cat_dims.append(len(enc.classes_))

    imputer = SimpleImputer(strategy='median')
    scaler  = StandardScaler()
    X_tr[num_cols] = imputer.fit_transform(X_tr[num_cols])
    X_te[num_cols] = imputer.transform(X_te[num_cols])
    X_tr[num_cols] = scaler.fit_transform(X_tr[num_cols])
    X_te[num_cols] = scaler.transform(X_te[num_cols])

    return {'X_train_np': X_tr.values.astype(np.float32),
            'X_test_np':  X_te.values.astype(np.float32),
            'cat_idxs': list(range(len(cat_features))),
            'cat_dims': cat_dims}


def train_tabnet(X_train, y_train, X_test, y_test, cat_features):
    prep = build_tabnet_preprocessing(X_train, X_test, cat_features)
    n = len(X_train)
    batch_size         = min(TABNET_BATCH_MAX, max(32, 2 ** int(np.log2(max(n // 8, 32)))))
    virtual_batch_size = min(128, max(16, batch_size // 8))

    model = TabNetClassifier(
        cat_idxs=prep['cat_idxs'], cat_dims=prep['cat_dims'], cat_emb_dim=4,
        n_d=16, n_a=16, n_steps=3, gamma=1.3, lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=2e-2),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params=dict(step_size=15, gamma=0.9),
        seed=RANDOM_STATE,
        device_name='cuda',        # <- force CUDA
        verbose=10,                # print toutes les 10 epochs (preuve d'activité)
    )

    t0 = time.perf_counter()
    model.fit(X_train=prep['X_train_np'], y_train=y_train.values.astype(np.int64),
              eval_set=[(prep['X_test_np'], y_test.values.astype(np.int64))],
              eval_name=['test'], eval_metric=['auc'],
              max_epochs=TABNET_EPOCHS, patience=15,
              batch_size=batch_size, virtual_batch_size=virtual_batch_size,
              num_workers=0, drop_last=False, weights=1)
    elapsed = time.perf_counter() - t0

    y_proba = model.predict_proba(prep['X_test_np'])[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    return model, y_pred, y_proba, elapsed


def save_tabnet(model, path):
    model.save_model(str(path))

---
## 9. Registre des modèles

In [ ]:
MODEL_REGISTRY = {
    'CatBoost': dict(train_fn=train_catboost, save_fn=save_catboost),
    'XGBoost':  dict(train_fn=train_xgboost,  save_fn=save_xgboost),
    'TabNet':   dict(train_fn=train_tabnet,   save_fn=save_tabnet),
}
print('Modèles enregistrés :', list(MODEL_REGISTRY.keys()))

---
## 10. Chargement du test set

In [ ]:
test_path = DATA_DIR / TEST_FILE
X_test, y_test, cat_features_test = load_dataset(test_path)

print(f'Test set  : {test_path.name}')
print(f'  shape   : {X_test.shape}')
print(f'  fraude  : {y_test.mean()*100:.3f} %  ({int(y_test.sum())} positifs / {len(y_test)})')
print(f'  cat cols: {len(cat_features_test)}')
print(f'  mémoire : {X_test.memory_usage(deep=True).sum()/1024**2:.2f} MB')

---
## 11. Boucle principale avec **barre de progression en %** + monitoring VRAM

La barre principale `tqdm` compte `len(files) * 3` étapes (3 modèles par fichier) → l'avancement est affiché en **%** et une ETA est calculée automatiquement.

Pour chaque modèle, un `VRAMMonitor` tourne en arrière-plan et rapporte le **peak VRAM** à la fin → c'est la preuve vivante que le GPU bosse.

In [ ]:
def run_single_experiment(train_path, model_name, cfg, X_test, y_test, pbar):
    ratio = extract_ratio(train_path)
    X_train, y_train, cat_features = load_dataset(train_path)

    pbar.set_description(f'{train_path.name}  |  {model_name}')

    mon = VRAMMonitor(interval=2.0)
    mon.start()
    t_global = time.perf_counter()
    try:
        model, y_pred, y_proba, train_time = cfg['train_fn'](
            X_train, y_train, X_test, y_test, cat_features)
    finally:
        mon.stop()

    metrics = compute_metrics(y_test.values, y_pred, y_proba)

    # Preuves GPU : VRAM peak + nvidia-smi snapshot
    peak_alloc_mb = max([s[1] for s in mon.samples], default=0.0)
    peak_reserved_mb = max([s[2] for s in mon.samples], default=0.0)

    ratio_tag  = f'{ratio:05.1f}'.replace('.', '_')
    model_path = MODELS_DIR / f'{model_name.lower()}_train_{ratio_tag}'
    cfg['save_fn'](model, model_path)

    row = {
        'Model':                 model_name,
        'Dataset_Ratio':         ratio,
        'Dataset_File':          Path(train_path).name,
        'N_Train':               len(X_train),
        'Fraud_Rate_Train_%':    round(y_train.mean()*100, 4),
        **{k: round(v, 6) if isinstance(v, float) else v for k, v in metrics.items()},
        'Training_Time_Seconds': round(train_time, 2),
        'Total_Time_Seconds':    round(time.perf_counter() - t_global, 2),
        'VRAM_Peak_MB':          round(peak_alloc_mb, 1),
        'VRAM_Reserved_Peak_MB': round(peak_reserved_mb, 1),
    }
    append_result_row(row, RESULTS_CSV)

    pbar.write(
        f'  {model_name:<8} | F1={metrics["F1"]:.3f} | PR-AUC={metrics["PR-AUC"]:.3f} | '
        f'Recall={metrics["Recall"]:.3f} | VRAM peak={peak_alloc_mb:>5.0f} MB | '
        f'time={train_time:5.1f}s'
    )

    del model, X_train, y_train, y_pred, y_proba
    gc.collect(); torch.cuda.empty_cache()
    return row

In [ ]:
if RESULTS_CSV.exists():
    RESULTS_CSV.unlink()

all_train_files = sorted(DATA_DIR.glob(TRAIN_GLOB))
if RUN_MODE == 'smoke':
    train_files = [f for f in all_train_files if extract_ratio(f) in SMOKE_RATIOS]
else:
    train_files = all_train_files

print(f'=== {RUN_MODE.upper()} MODE  —  {len(train_files)} fichier(s) × {len(MODEL_REGISTRY)} modèles = '
      f'{len(train_files)*len(MODEL_REGISTRY)} expériences ===\n')

total_steps = len(train_files) * len(MODEL_REGISTRY)
t_all = time.perf_counter()
all_results = []

with tqdm(total=total_steps, desc='Training', unit='exp',
          bar_format='{l_bar}{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}] {postfix}'
         ) as pbar:
    for train_path in train_files:
        ratio = extract_ratio(train_path)
        pbar.write(f'\n═══ {train_path.name}  (ratio={ratio:.1f}%) ═══')
        for model_name, cfg in MODEL_REGISTRY.items():
            try:
                row = run_single_experiment(train_path, model_name, cfg, X_test, y_test, pbar)
                all_results.append(row)
            except Exception as e:
                pbar.write(f'  {model_name:<8} | ❌ ERREUR : {type(e).__name__}: {e}')
                append_result_row({'Model': model_name, 'Dataset_Ratio': ratio,
                                    'Dataset_File': train_path.name,
                                    'Error': f'{type(e).__name__}: {e}'}, RESULTS_CSV)
            pbar.update(1)
            pbar.set_postfix({'last_f1': f"{row['F1']:.3f}" if 'row' in dir() else 'n/a'})

print(f'\n=== TERMINÉ en {(time.perf_counter()-t_all)/60:.1f} min ===')
print(f'Résultats : {RESULTS_CSV.resolve()}')

---
## 12. Preuve GPU finale — nvidia-smi snapshot après les runs

In [ ]:
!nvidia-smi

# Si tu vois dans le CSV que VRAM_Peak_MB > 200-500 MB pour chaque modèle, le GPU a bien été sollicité.
# Sur CPU ces mêmes runs afficheraient VRAM_Peak_MB ≈ 0.

---
## 13. Analyse des résultats

In [ ]:
results = pd.read_csv(RESULTS_CSV)
print(f'Résultats : {results.shape}')
results

In [ ]:
if 'PR-AUC' in results.columns:
    lb = (results.dropna(subset=['PR-AUC'])
                 .sort_values('PR-AUC', ascending=False)
                 .loc[:, ['Model', 'Dataset_Ratio', 'F1', 'Recall', 'Precision',
                          'PR-AUC', 'ROC-AUC', 'TP', 'FP', 'FN', 'TN',
                          'Training_Time_Seconds', 'VRAM_Peak_MB']]
                 .reset_index(drop=True))
    print('=== TOP 10 par PR-AUC ===')
    print(lb.head(10).to_string(index=False))
    print('\n=== Meilleur résultat par modèle ===')
    best_per_model = lb.loc[lb.groupby('Model')['PR-AUC'].idxmax()].sort_values('PR-AUC', ascending=False)
    print(best_per_model.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = ['F1', 'Recall', 'Precision', 'PR-AUC']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, m in zip(axes.ravel(), metrics_to_plot):
    for model_name in results['Model'].dropna().unique():
        sub = (results[results['Model'] == model_name]
               .dropna(subset=[m]).sort_values('Dataset_Ratio'))
        ax.plot(sub['Dataset_Ratio'], sub[m], marker='o', lw=2, label=model_name)
    ax.set_xlabel('Ratio undersampling train (% de fraude)')
    ax.set_ylabel(m); ax.set_title(f'{m} vs ratio d\'undersampling')
    ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig(LOG_DIR / 'metrics_vs_ratio.png', dpi=110, bbox_inches='tight')
plt.show()

---
## 14. Télécharger les résultats

Sur Colab, tu peux récupérer l'ensemble des logs en un clic.

In [ ]:
# Zip tout le dossier logs_training_colab pour un download facile
!zip -r logs_training_colab.zip logs_training_colab > /dev/null
print('Archive créée : logs_training_colab.zip')

# Décommente pour télécharger :
# from google.colab import files
# files.download('logs_training_colab.zip')